# HW13: Intro to NLP, pt 1: Tf-idf

In [1]:
import re
import logging
import time

from bs4 import BeautifulSoup
from gensim.models import Word2Vec
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, \
    TfidfTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, \
    GridSearchCV
from sklearn.pipeline import Pipeline

In [2]:
%load_ext pycodestyle_magic

In [3]:
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s',
                    level=logging.INFO)
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /home/anton/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/anton/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/anton/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## Import data

In [4]:
sentiment_data = pd.read_csv('data/train_data.tsv', header=0,
                             delimiter='\t', quoting=3)
train_data, test_data = train_test_split(sentiment_data,
                                         test_size=0.2, random_state=42)

In [5]:
sentiment_data.head()

,id,sentiment,review
0,"""5814_8""",1,"""With all this stuff going down at the moment ..."
1,"""2381_9""",1,"""\""The Classic War of the Worlds\"" by Timothy ..."
2,"""7759_3""",0,"""The film starts with a manager (Nicholas Bell..."
3,"""3630_4""",0,"""It must be assumed that those who praised thi..."
4,"""9495_8""",1,"""Superbly trashy and wondrously unpretentious ..."


In [6]:
train_data, test_data = train_test_split(sentiment_data,
                                         test_size=0.2, random_state=42)

### Defining some helper functions

In [7]:
wordnet_lemmatizer = WordNetLemmatizer()

In [8]:
def series_to_wordlist(series):

    train_text0 = series.apply(lambda text:
                               BeautifulSoup(text, 'html.parser').get_text())
    train_text0 = train_text0.str.replace("[^a-zA-Z]", " ").str.lower()
    train_words = train_text0.str.split()

    stops = set(stopwords.words("english"))
    train_words = train_words.apply(lambda text:
                                    " ".join([wordnet_lemmatizer.lemmatize(t, 'v')
                                     for t in text if t not in stops]))

    return train_words.values.ravel()

### Reviews go to sentences

In [9]:
%%time
sentences = series_to_wordlist(train_data['review'])

CPU times: user 12.4 s, sys: 140 ms, total: 12.6 s
Wall time: 12.6 s


In [10]:
len(sentences)

20000

In [11]:
print(sentences[0])

movie plain dumb cast ralph meeker mike hammer fatuous climax film exercise wooden predictability mike hammer one detective fiction true sociopaths unlike marlow spade put piece together solve mystery hammer break things apart get truth film turn hammer boob surround bad guy well dumb get away anything one poorly draw succumb popcorn attack part movie right three stooge play book velda dance barre instance bad guy accidentally stab boss back continuity break shameful frau blucher run centerline road camera tight lower legs way side camera pull back wider shoot worst break however precede popcorn attack bad guy stalk hammer pass clock second hero except clock show seven minutes behind guy fair interest camera angle light grand finale bad must see reason get two point


### TF-Idf

In [12]:
vectorizer = TfidfVectorizer(max_features=5000)
train_data_features = vectorizer.fit_transform(sentences)
train_data_features = train_data_features.toarray()

In [13]:
train_data_features.shape

(20000, 5000)

In [14]:
vocab = vectorizer.get_feature_names()
vocab[:10]

['abandon',
 'abc',
 'abilities',
 'ability',
 'able',
 'abound',
 'abraham',
 'abrupt',
 'absence',
 'absent']

### Classification

In [15]:
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1)
rf.fit(train_data_features, train_data['sentiment'])

RandomForestClassifier(bootstrap=True, class_weight=None, criterion='gini',
                       max_depth=None, max_features='auto', max_leaf_nodes=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=100,
                       n_jobs=-1, oob_score=False, random_state=None, verbose=0,
                       warm_start=False)

In [16]:
print('Parsing sentences from test set')

test_sentences = series_to_wordlist(test_data['review'])

Parsing sentences from test set


In [17]:
test_data_features = vectorizer.transform(test_sentences).toarray()

In [18]:
roc_auc_score(test_data['sentiment'],
              rf.predict_proba(test_data_features)[:, 1])

0.9242097023524078

### Now that there is a result, trying to make it a "proper" way

In [19]:
pipeline = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', RandomForestClassifier()),
])

In [20]:
parameters = {
    'vect__max_df': [0.5, 0.75, 1.0],
    'vect__max_features': [None, 1000, 5000, 10000, 50000],
    'vect__ngram_range': [(1, 1), (1, 2)],  # unigrams or bigrams
    'tfidf__use_idf': [True, False],
    'tfidf__norm': ['l1', 'l2'],
    'clf__n_estimators': [100],
    'clf__max_depth': [None]
}

In [21]:
grid_search = RandomizedSearchCV(pipeline, parameters, cv=5,
                                 n_jobs=-1, verbose=100, scoring='roc_auc')
grid_search.fit(sentences, train_data['sentiment'])

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
Pickling array (shape=(20000,), dtype=object).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(15999,), dtype=int64).
Pickling array (shape=(4001,), dtype=int64).
Pickling array (shape=(20000,), dtype=object).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(16000,), dtype=int64).
Pickling array (shape=(4000,), dtype=int64).
Pickling array (shape=(20000,), dtype=object).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(16000,), dtype=int64).
Pickling array (shape=(4000,), dtype=int64).
Pickling array (shape=(20000,), dty

[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:  2.1min
Pickling array (shape=(20000,), dtype=object).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(16000,), dtype=int64).
Pickling array (shape=(4000,), dtype=int64).
[Parallel(n_jobs=-1)]: Done  19 tasks      | elapsed:  2.1min
Pickling array (shape=(20000,), dtype=object).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(16000,), dtype=int64).
Pickling array (shape=(4000,), dtype=int64).
[Parallel(n_jobs=-1)]: Done  20 tasks      | elapsed:  2.2min
Pickling array (shape=(20000,), dtype=object).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(20000,), dtype=int64).
Pickling array (shape=(16000,), dtype=int64).
Pickling array (shape=(4000,), 

[Parallel(n_jobs=-1)]: Done  42 out of  50 | elapsed:  4.5min remaining:   51.1s
[Parallel(n_jobs=-1)]: Done  43 out of  50 | elapsed:  4.5min remaining:   43.8s
[Parallel(n_jobs=-1)]: Done  44 out of  50 | elapsed:  4.6min remaining:   37.5s
[Parallel(n_jobs=-1)]: Done  45 out of  50 | elapsed:  4.6min remaining:   30.6s
[Parallel(n_jobs=-1)]: Done  46 out of  50 | elapsed:  7.9min remaining:   41.1s
[Parallel(n_jobs=-1)]: Done  47 out of  50 | elapsed:  8.4min remaining:   32.3s
[Parallel(n_jobs=-1)]: Done  48 out of  50 | elapsed:  8.7min remaining:   21.7s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:  8.9min remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:  8.9min finished


RandomizedSearchCV(cv=5, error_score='raise-deprecating',
                   estimator=Pipeline(memory=None,
                                      steps=[('vect',
                                              CountVectorizer(analyzer='word',
                                                              binary=False,
                                                              decode_error='strict',
                                                              dtype=<class 'numpy.int64'>,
                                                              encoding='utf-8',
                                                              input='content',
                                                              lowercase=True,
                                                              max_df=1.0,
                                                              max_features=None,
                                                              min_df=1,
                                          

In [22]:
grid_search.best_params_

{'vect__ngram_range': (1, 2),
 'vect__max_features': 50000,
 'vect__max_df': 1.0,
 'tfidf__use_idf': False,
 'tfidf__norm': 'l1',
 'clf__n_estimators': 100,
 'clf__max_depth': None}

In [23]:
grid_search.best_score_

0.9292142552776899

In [24]:
roc_auc_score(test_data['sentiment'],
              grid_search.best_estimator_.predict_proba(test_sentences)[:, 1])

0.9335097595237102

A bit better now